In [167]:
import urllib.request
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/karpathy/makemore/master/names.txt',
    'names.txt'
)

('names.txt', <http.client.HTTPMessage at 0x1aa23e2d370>)

In [168]:
# 2147483647
import torch

In [169]:
words = open('names.txt', 'r').read().splitlines()

In [170]:
training_size = int(len(words)*0.8)
dev_size = int(len(words)*0.1)
test_size = len(words) - training_size - dev_size

In [171]:
from torch.utils.data import random_split
g = torch.Generator().manual_seed(2147483647)
train_set, dev_set, test_set = random_split(words, [training_size, dev_size, test_size], generator = g)

In [172]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}

In [173]:
xs = [] 
ys = []
for w in train_set:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        ix3 = stoi[ch3]
        xs.append(ix1*27 + ix2)
        ys.append(ix3)

xs = torch.tensor(xs)
ys = torch.tensor(ys)

In [174]:
xs.shape

torch.Size([156957])

In [249]:
import torch.nn.functional as F
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((729, 27), dtype = torch.float32, generator = g, requires_grad = True)

In [250]:
for i in range(200):
    # logits = xenc @ W
    logits = W[xs] # equivalent to logits[i] = W[xs[i]] for i in range(len(xs))
    # counts = logits.exp()
    # probs = counts/counts.sum(1, keepdims = True)
    # loss = (-probs[torch.arange(len(xs)), ys].log()).mean() + 0.001*(W**2).mean()
    loss = F.cross_entropy(logits, ys)

    W.grad = None
    loss.backward()

    W.data += -85*W.grad

print(loss)

tensor(2.1912, grad_fn=<NllLossBackward0>)


In [177]:
g = torch.Generator().manual_seed(2147483647)
for i in range(5):
    out = []
    ix = 0
    prev_index = 0
    while True:
        xenc = F.one_hot(torch.tensor([ix]), num_classes = 729).float()
        logits = xenc @ W
        counts = logits.exp()
        probs = counts / counts.sum(1, keepdims = True)
        index = torch.multinomial(probs, num_samples = 1, replacement = True, generator = g).item()
        out.append(itos[index])
        if index == 0:
            break
        ix = prev_index*27 + index
        prev_index = index
    word = ''.join(out)
    print(word)

zexzdfzjglkuriana.
ottharlinichana.
orakayk.
ka.
or.


In [251]:
x_dev = []
y_dev = []
for w in dev_set:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        ix3 = stoi[ch3]
        x_dev.append(ix1*27 + ix2)
        y_dev.append(ix3)

x_dev = torch.tensor(x_dev)
y_dev = torch.tensor(y_dev)

In [252]:
xenc_dev = F.one_hot(x_dev, num_classes = 729).float()

In [253]:
logits = xenc_dev @ W
counts = logits.exp()
probs = counts / counts.sum(1, keepdims = True)
loss = (-probs[torch.arange(len(x_dev)),y_dev].log()).mean()
loss

tensor(2.2145, grad_fn=<MeanBackward0>)

In [254]:
x_test = []
y_test = []
for w in test_set:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        ix3 = stoi[ch3]
        x_test.append(ix1*27 + ix2)
        y_test.append(ix3)

x_test = torch.tensor(x_test)
y_test = torch.tensor(y_test)

In [255]:
xenc_test = F.one_hot(x_test, num_classes = 729).float()

In [256]:
logits = xenc_test @ W
counts = logits.exp()
probs = counts / counts.sum(1, keepdims = True)
loss = (-probs[torch.arange(len(x_test)), y_test].log()).mean()
loss

tensor(2.2056, grad_fn=<MeanBackward0>)